# 009_decoding_plots_condition_specific_outputs.ipynb

Paper-ready decoding plots for the condition-specific ranking branch. Loads `msaa_condrank_decoding_outputs_{analysis_type}_{fit_scope}`.


In [ ]:
# ============================================================
# SETTINGS
# ============================================================

# ANALYSIS_TYPES = ["spatial", "temporal"]
# FIT_SCOPES = ["within", "across"]

ANALYSIS_TYPES = ["spatial"]
FIT_SCOPES = ["across"]

DECODE_DIR_TEMPLATE = "msaa_condrank_decoding_outputs_{analysis_type}_{fit_scope}"
FIGURE_OUT_DIR = "009_paper_decoding_figures_condition_specific"

SAVE_FIGS = True
FIG_FORMAT = "pdf"
DPI = 300

# None = all available K values
# K_VALUES_TO_PLOT = None
# Example:
K_VALUES_TO_PLOT = [35, 40, 45, 50, 55, 60, 65]

CONDITIONS = ["intact", "word", "rest"]
COND_COLORS = {"intact": "purple", "word": "green", "rest": "black"}

# None = all available top_m values
TOP_M_VALUES_TO_PLOT = None
# Example:
# TOP_M_VALUES_TO_PLOT = [1, 3, 5, 10]

FIGSIZE_SINGLE = (7.0, 4.8)
FIGSIZE_WIDE = (8.8, 5.2)

FONT_SIZE = 12
TITLE_SIZE = 13
LABEL_SIZE = 12
TICK_SIZE = 10
LEGEND_SIZE = 10

LINEWIDTH = 2.2
MARKER_SIZE = 6
CAPSIZE = 4

YLIM = None

SHOW_CHANCE_LINE = False
CHANCE_LEVEL = None

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

%matplotlib inline

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Path(FIGURE_OUT_DIR).mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": FONT_SIZE,
    "axes.titlesize": TITLE_SIZE,
    "axes.labelsize": LABEL_SIZE,
    "xtick.labelsize": TICK_SIZE,
    "ytick.labelsize": TICK_SIZE,
    "legend.fontsize": LEGEND_SIZE,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("Ready.")
print("Figure output directory:", FIGURE_OUT_DIR)

In [ ]:
# ============================================================
# FIGURE SAVING SETTINGS -- EXPLICIT, NO RECURSION
# ============================================================

from pathlib import Path

SAVE_FIGS = True
FIG_ROOT = "/Users/lowen/Desktop/papers/archetypes/figures"
FIG_NOTEBOOK_DIR = "009_decoding_plots_outputs"
FIG_DIR = Path(FIG_ROOT) / FIG_NOTEBOOK_DIR
FIG_FORMAT = "pdf"   # "pdf", "png", or "svg"
DPI = 300

FIG_DIR.mkdir(parents=True, exist_ok=True)

_fig_counter = 0

def _sanitize_fig_name(name):
    name = str(name).replace(" ", "_").replace("|", "_").replace("/", "-").replace("\\", "-")
    name = "".join(ch for ch in name if ch.isalnum() or ch in ["_", "-", "."])
    return name[:160] if name else "figure"

def _figure_has_content(fig=None):
    if fig is None:
        fig = plt.gcf()
    if len(fig.axes) == 0:
        return False
    for ax in fig.axes:
        if ax.lines or ax.collections or ax.images or ax.patches or ax.texts or ax.get_title():
            return True
    return True

def save_current_fig(name=None):
    global _fig_counter
    if not SAVE_FIGS:
        return None
    fig = plt.gcf()
    if not _figure_has_content(fig):
        return None
    _fig_counter += 1
    if name is None:
        try:
            title = plt.gca().get_title()
        except Exception:
            title = ""
        label = _sanitize_fig_name(title if title else "figure")
    else:
        label = _sanitize_fig_name(name)
    out = FIG_DIR / f"{_fig_counter:03d}_{label}.{FIG_FORMAT}"
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    print("Saved:", out)
    return out

def savefig(name=None, force=True):
    return save_current_fig(name=name)

print("Figure directory:", FIG_DIR)
print("Explicit save mode: plt.show is not patched.")


In [ ]:
# ============================================================
# CACHE SETTINGS
# ============================================================

from pathlib import Path
import pickle

CACHE_DIR = Path("009_decoding_plots_outputs_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

USE_CACHE = True
OVERWRITE_CACHE = False

CLUSTER_SUMMARY_CACHE = CACHE_DIR / "cluster_summary_df.csv"
SELECTED_ARCHETYPES_CACHE = CACHE_DIR / "selected_archetypes_dict.npy"
PLOT_DATA_CACHE = CACHE_DIR / "plot_data_cache.pkl"

print("Cache directory:", CACHE_DIR.resolve())
print("USE_CACHE:", USE_CACHE)
print("OVERWRITE_CACHE:", OVERWRITE_CACHE)

## Helper functions

In [ ]:
def savefig(name):
    if not SAVE_FIGS:
        return
    out = Path(FIGURE_OUT_DIR) / f"{name}.{FIG_FORMAT}"
    plt.savefig(out, dpi=DPI, bbox_inches="tight")
    print("Saved:", out)


def maybe_chance_and_ylim():
    if SHOW_CHANCE_LINE and CHANCE_LEVEL is not None:
        plt.axhline(CHANCE_LEVEL, color="gray", linestyle="--", linewidth=1)
    if YLIM is not None:
        plt.ylim(*YLIM)


def load_csv(path):
    path = Path(path)
    if path.exists():
        print("Loaded:", path)
        return pd.read_csv(path)
    return None


def standardize_decoding_df(df, analysis_type=None, fit_scope=None):
    if df is None:
        return None

    df = df.copy()

    rename_map = {}
    if "mean_accuracy" in df.columns and "mean" not in df.columns:
        rename_map["mean_accuracy"] = "mean"
    if "sem_accuracy" in df.columns and "err" not in df.columns:
        rename_map["sem_accuracy"] = "err"
    if "std_accuracy" in df.columns and "std" not in df.columns:
        rename_map["std_accuracy"] = "std"
    if "sem" in df.columns and "err" not in df.columns:
        rename_map["sem"] = "err"
    if "stderr" in df.columns and "err" not in df.columns:
        rename_map["stderr"] = "err"
    if "accuracy" in df.columns and "mean" not in df.columns:
        rename_map["accuracy"] = "mean"

    df = df.rename(columns=rename_map)

    if analysis_type is not None and "analysis_type" not in df.columns:
        df["analysis_type"] = analysis_type
    if fit_scope is not None and "fit_scope" not in df.columns:
        df["fit_scope"] = fit_scope

    if "K" in df.columns:
        df["K"] = df["K"].astype(int)
    if "top_m" in df.columns:
        df["top_m"] = df["top_m"].astype(int)
    if "archetype" in df.columns:
        df["archetype"] = df["archetype"].astype(int)
    if "condition" in df.columns:
        df["condition"] = df["condition"].astype(str)

    return df


def filter_df(df, analysis_type=None, fit_scope=None):
    if df is None:
        return None
    out = df.copy()

    if analysis_type is not None and "analysis_type" in out.columns:
        out = out[out["analysis_type"].astype(str) == str(analysis_type)]
    if fit_scope is not None and "fit_scope" in out.columns:
        out = out[out["fit_scope"].astype(str) == str(fit_scope)]
    if K_VALUES_TO_PLOT is not None and "K" in out.columns:
        out = out[out["K"].isin(K_VALUES_TO_PLOT)]

    return out.copy()


def get_err_col(df):
    if df is None:
        return None
    for col in ["err", "sem", "stderr", "se", "sem_accuracy"]:
        if col in df.columns:
            return col
    return None


def validate_for_plot(df, required_cols, label):
    if df is None:
        print(f"Skipping {label}: dataframe is None.")
        return False
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        print(f"Skipping {label}: missing columns {missing}. Available columns: {list(df.columns)}")
        return False
    if len(df) == 0:
        print(f"Skipping {label}: dataframe has 0 rows after filtering.")
        return False
    return True

## Load saved appendable decoding outputs

In [ ]:
def load_decoding_outputs(analysis_type, fit_scope):
    d = Path(DECODE_DIR_TEMPLATE.format(analysis_type=analysis_type, fit_scope=fit_scope))

    full_summary = load_csv(d / "full_summary.csv")
    topm_summary = load_csv(d / "topm_summary.csv")
    per_archetype = load_csv(d / "per_archetype_mean_accuracy.csv")

    if full_summary is None and topm_summary is None and per_archetype is None:
        print(f"WARNING: no decoding CSVs found for {analysis_type}-{fit_scope} in {d}")
        if d.exists():
            print("Files in directory:")
            for p in sorted(d.glob("*")):
                print("  ", p.name)
        else:
            print("Directory does not exist:", d)

    return {
        "decode_dir": d,
        "full_summary": filter_df(standardize_decoding_df(full_summary, analysis_type, fit_scope), analysis_type, fit_scope),
        "topm_summary": filter_df(standardize_decoding_df(topm_summary, analysis_type, fit_scope), analysis_type, fit_scope),
        "per_archetype": filter_df(standardize_decoding_df(per_archetype, analysis_type, fit_scope), analysis_type, fit_scope),
    }


all_outputs = {}
for analysis_type in ANALYSIS_TYPES:
    for fit_scope in FIT_SCOPES:
        print("\n" + "=" * 80)
        print(f"Loading {analysis_type} | {fit_scope}")
        print("=" * 80)
        all_outputs[(analysis_type, fit_scope)] = load_decoding_outputs(analysis_type, fit_scope)

# Plot 1: Full reconstruction decoding curves

In [ ]:
def plot_full_curves(analysis_type, fit_scope, df):
    label = f"{analysis_type}-{fit_scope} full"
    if not validate_for_plot(df, ["condition", "K", "mean"], label):
        return

    err_col = get_err_col(df)
    plt.figure(figsize=FIGSIZE_SINGLE)

    for cond in CONDITIONS:
        sub = df[df["condition"].astype(str) == cond].sort_values("K")
        if len(sub) == 0:
            continue
        yerr = sub[err_col] if err_col is not None else None
        plt.errorbar(
            sub["K"], sub["mean"], yerr=yerr,
            marker="o", markersize=MARKER_SIZE,
            linewidth=LINEWIDTH, capsize=CAPSIZE,
            color=COND_COLORS[cond], label=cond
        )

    maybe_chance_and_ylim()
    plt.xlabel("Number of archetypes (K)")
    plt.ylabel("Decoding accuracy")
    plt.title(f"{analysis_type.capitalize()} AA | {fit_scope} | full reconstruction")
    plt.legend(frameon=False)
    plt.tight_layout()
    savefig(f"decoding_full_{analysis_type}_{fit_scope}")
    save_current_fig()
    plt.show()
    plt.close()
for (analysis_type, fit_scope), out in all_outputs.items():
    plot_full_curves(analysis_type, fit_scope, out["full_summary"])

# Plot 2: Top-m decoding curves

In [ ]:
def plot_topm_conditions_one_plot(analysis_type, fit_scope, df):
    label = f"{analysis_type}-{fit_scope} topm"
    if not validate_for_plot(df, ["condition", "K", "top_m", "mean"], label):
        return

    err_col = get_err_col(df)
    top_m_values = sorted(df["top_m"].dropna().unique())

    if TOP_M_VALUES_TO_PLOT is not None:
        top_m_values = [m for m in top_m_values if m in TOP_M_VALUES_TO_PLOT]

    for top_m in top_m_values:
        plt.figure(figsize=FIGSIZE_SINGLE)

        for cond in CONDITIONS:
            sub = df[(df["condition"].astype(str) == cond) & (df["top_m"] == top_m)].sort_values("K")
            if len(sub) == 0:
                continue
            yerr = sub[err_col] if err_col is not None else None
            plt.errorbar(
                sub["K"], sub["mean"], yerr=yerr,
                marker="o", markersize=MARKER_SIZE,
                linewidth=LINEWIDTH, capsize=CAPSIZE,
                color=COND_COLORS[cond], label=cond
            )

        maybe_chance_and_ylim()
        plt.xlabel("Number of archetypes (K)")
        plt.ylabel("Decoding accuracy")
        plt.title(f"{analysis_type.capitalize()} AA | {fit_scope} | top-{top_m} reconstruction")
        plt.legend(frameon=False)
        plt.tight_layout()
        savefig(f"decoding_top{top_m}_{analysis_type}_{fit_scope}")
        save_current_fig()
        plt.show()
        plt.close()
for (analysis_type, fit_scope), out in all_outputs.items():
    plot_topm_conditions_one_plot(analysis_type, fit_scope, out["topm_summary"])

# Plot 3: Spatial vs temporal comparison

In [ ]:
def plot_spatial_vs_temporal(fit_scope, use="full", top_m=None):
    plt.figure(figsize=FIGSIZE_WIDE)
    linestyles = {"spatial": "--", "temporal": "-"}
    suffix = "full" if use == "full" else f"top-{top_m}"
    any_plotted = False

    for analysis_type in ["spatial", "temporal"]:
        out = all_outputs.get((analysis_type, fit_scope))
        if out is None:
            continue

        if use == "full":
            df = out["full_summary"]
        elif use == "topm":
            df = out["topm_summary"]
            if df is not None and "top_m" in df.columns:
                top_m_local = sorted(df["top_m"].unique())[0] if top_m is None else top_m
                df = df[df["top_m"] == top_m_local]
                suffix = f"top-{top_m_local}"
        else:
            raise ValueError("use must be 'full' or 'topm'")

        if not validate_for_plot(df, ["condition", "K", "mean"], f"{analysis_type}-{fit_scope}-{use}"):
            continue

        err_col = get_err_col(df)

        for cond in CONDITIONS:
            sub = df[df["condition"].astype(str) == cond].sort_values("K")
            if len(sub) == 0:
                continue
            yerr = sub[err_col] if err_col is not None else None
            plt.errorbar(
                sub["K"], sub["mean"], yerr=yerr,
                marker="o", markersize=MARKER_SIZE,
                linewidth=LINEWIDTH, capsize=CAPSIZE,
                color=COND_COLORS[cond],
                linestyle=linestyles[analysis_type],
                label=f"{analysis_type} | {cond}"
            )
            any_plotted = True

    if not any_plotted:
        print(f"No data plotted for spatial vs temporal: {fit_scope} {use}")
        return

    maybe_chance_and_ylim()
    plt.xlabel("Number of archetypes (K)")
    plt.ylabel("Decoding accuracy")
    plt.title(f"Spatial vs temporal AA | {fit_scope} | {suffix}")
    plt.legend(frameon=False, ncol=2)
    plt.tight_layout()
    savefig(f"decoding_spatial_vs_temporal_{fit_scope}_{suffix}")
    save_current_fig()
    plt.show()
    plt.close()
for fit_scope in FIT_SCOPES:
    plot_spatial_vs_temporal(fit_scope, use="full")

# Plot 4: Within vs across comparison

In [ ]:
def plot_within_vs_across(analysis_type, use="full", top_m=None):
    plt.figure(figsize=FIGSIZE_WIDE)
    linestyles = {"within": "-", "across": "--"}
    suffix = "full" if use == "full" else f"top-{top_m}"
    any_plotted = False

    for fit_scope in ["within", "across"]:
        out = all_outputs.get((analysis_type, fit_scope))
        if out is None:
            continue

        if use == "full":
            df = out["full_summary"]
        elif use == "topm":
            df = out["topm_summary"]
            if df is not None and "top_m" in df.columns:
                top_m_local = sorted(df["top_m"].unique())[0] if top_m is None else top_m
                df = df[df["top_m"] == top_m_local]
                suffix = f"top-{top_m_local}"
        else:
            raise ValueError("use must be 'full' or 'topm'")

        if not validate_for_plot(df, ["condition", "K", "mean"], f"{analysis_type}-{fit_scope}-{use}"):
            continue

        err_col = get_err_col(df)

        for cond in CONDITIONS:
            sub = df[df["condition"].astype(str) == cond].sort_values("K")
            if len(sub) == 0:
                continue
            yerr = sub[err_col] if err_col is not None else None
            plt.errorbar(
                sub["K"], sub["mean"], yerr=yerr,
                marker="o", markersize=MARKER_SIZE,
                linewidth=LINEWIDTH, capsize=CAPSIZE,
                color=COND_COLORS[cond],
                linestyle=linestyles[fit_scope],
                label=f"{fit_scope} | {cond}"
            )
            any_plotted = True

    if not any_plotted:
        print(f"No data plotted for within vs across: {analysis_type} {use}")
        return

    maybe_chance_and_ylim()
    plt.xlabel("Number of archetypes (K)")
    plt.ylabel("Decoding accuracy")
    plt.title(f"Within vs across | {analysis_type} AA | {suffix}")
    plt.legend(frameon=False, ncol=2)
    plt.tight_layout()
    savefig(f"decoding_within_vs_across_{analysis_type}_{suffix}")
    save_current_fig()
    plt.show()
    plt.close()
for analysis_type in ANALYSIS_TYPES:
    plot_within_vs_across(analysis_type, use="full")

# Plot 5: Top-m grid

In [ ]:
def plot_topm_grid(analysis_type, fit_scope, df):
    label = f"{analysis_type}-{fit_scope} topm grid"
    if not validate_for_plot(df, ["condition", "K", "top_m", "mean"], label):
        return

    top_m_values = sorted(df["top_m"].dropna().unique())
    if TOP_M_VALUES_TO_PLOT is not None:
        top_m_values = [m for m in top_m_values if m in TOP_M_VALUES_TO_PLOT]
    if len(top_m_values) == 0:
        return

    n = len(top_m_values)
    fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 4.0), sharey=True)
    if n == 1:
        axes = [axes]

    err_col = get_err_col(df)

    for ax, top_m in zip(axes, top_m_values):
        for cond in CONDITIONS:
            sub = df[(df["condition"].astype(str) == cond) & (df["top_m"] == top_m)].sort_values("K")
            if len(sub) == 0:
                continue
            yerr = sub[err_col] if err_col is not None else None
            ax.errorbar(
                sub["K"], sub["mean"], yerr=yerr,
                marker="o", markersize=MARKER_SIZE,
                linewidth=LINEWIDTH, capsize=CAPSIZE,
                color=COND_COLORS[cond],
                label=cond
            )

        if SHOW_CHANCE_LINE and CHANCE_LEVEL is not None:
            ax.axhline(CHANCE_LEVEL, color="gray", linestyle="--", linewidth=1)
        if YLIM is not None:
            ax.set_ylim(*YLIM)

        ax.set_title(f"top-{top_m}")
        ax.set_xlabel("K")

    axes[0].set_ylabel("Decoding accuracy")
    axes[-1].legend(frameon=False)
    fig.suptitle(f"{analysis_type.capitalize()} AA | {fit_scope} | top-m reconstructions", y=1.05)
    plt.tight_layout()
    savefig(f"decoding_topm_grid_{analysis_type}_{fit_scope}")
    save_current_fig()
    plt.show()
    plt.close()
for (analysis_type, fit_scope), out in all_outputs.items():
    plot_topm_grid(analysis_type, fit_scope, out["topm_summary"])

## Inspect loaded tables

In [ ]:
for key, out in all_outputs.items():
    print("\n" + "="*80)
    print(key)
    print("="*80)
    for name in ["full_summary", "topm_summary", "per_archetype"]:
        df = out[name]
        if df is None:
            print(f"{name}: None")
        else:
            print(f"{name}: shape={df.shape}, columns={list(df.columns)}")
            display(df.head())